# Extrair pose de uma imagem com o ViTPose (branch `extracao-vitpose`)

Usa a classe `VitPoseDetector` de `experiments/compare_pose_detectors.py`, sem alterá-la:
RT-DETR encontra até 2 pessoas e o ViTPose+ huge estima 17 pontos COCO de cada uma.

Pré-requisitos: branch `extracao-vitpose` em checkout, este notebook dentro de `notebooks/`
e o kernel **JiujitsuCompVision (.venv)**. A primeira execução baixa os pesos do Hugging Face (alguns GB).

In [ ]:
import json
import sys
from pathlib import Path

# Raiz do repositório (este notebook fica em notebooks/)
ROOT = Path.cwd().resolve()
if not (ROOT / "experiments").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "experiments" / "compare_pose_detectors.py").is_file(), \
    "Não achei experiments/compare_pose_detectors.py: faça checkout da branch extracao-vitpose"
sys.path.insert(0, str(ROOT))

import torch
from PIL import Image, ImageDraw
from experiments.compare_pose_detectors import VitPoseDetector

print("Raiz:", ROOT, "| PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## Carregar o detector (uma vez só)
Use `"cuda"` se tiver GPU, senão `"cpu"`.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
detector = VitPoseDetector(DEVICE)

## Função de extração

`extrair_pose(caminho)` devolve um dicionário pronto para virar JSON. Cada pose tem 17 pontos
`[x, y, confiança]` em pixels da imagem original, na ordem COCO (`JOINTS`). É o mesmo formato
de `pose1`/`pose2` nas anotações do dataset.

In [ ]:
JOINTS = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
          "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
          "left_wrist", "right_wrist", "left_hip", "right_hip",
          "left_knee", "right_knee", "left_ankle", "right_ankle"]


def extrair_pose(caminho, salvar_em=None):
    caminho = Path(caminho)
    poses = detector(caminho)  # até 2 poses, cada uma 17 x [x, y, score]
    with Image.open(caminho) as imagem:
        largura, altura = imagem.size
    resultado = {
        "image": caminho.name,
        "width": largura,
        "height": altura,
        "joints": JOINTS,
        "people_detected": len(poses),
        "pose1": poses[0] if len(poses) > 0 else None,
        "pose2": poses[1] if len(poses) > 1 else None,
    }
    if salvar_em:
        Path(salvar_em).parent.mkdir(parents=True, exist_ok=True)
        Path(salvar_em).write_text(json.dumps(resultado, indent=1))
    return resultado

## Desenhar o esqueleto (para conferir se a detecção ficou boa)

In [ ]:
SKELETON = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6),
            (5, 7), (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]


def desenhar_pose(caminho, resultado, min_score=0.3):
    with Image.open(caminho) as fonte:
        imagem = fonte.convert("RGB")
    tela = ImageDraw.Draw(imagem)
    r = max(2, round(max(imagem.size) / 250))
    for pose, cor in ((resultado["pose1"], "#ff3b30"), (resultado["pose2"], "#0a84ff")):
        if pose is None:
            continue
        for a, b in SKELETON:
            if pose[a][2] >= min_score and pose[b][2] >= min_score:
                tela.line([tuple(pose[a][:2]), tuple(pose[b][:2])], fill=cor, width=r)
        for x, y, s in pose:
            if s >= min_score:
                tela.ellipse([x - r, y - r, x + r, y + r], fill=cor)
    return imagem

## Usar em uma imagem
Troque o caminho pela sua imagem.

In [ ]:
# IMAGEM = ROOT / "minhas_imagens" / "exemplo.jpg"
IMAGEM = ROOT / "notebooks" / "i11.jpg"
SAIDA = ROOT / "diagnostics" / "poses" / f"{IMAGEM.stem}.json"

resultado = extrair_pose(IMAGEM, salvar_em=SAIDA)
print(f"{resultado['people_detected']} pessoa(s) detectada(s). JSON salvo em {SAIDA}")
desenhar_pose(IMAGEM, resultado)

In [ ]:
# Ver o JSON
print(json.dumps(resultado, indent=1)[:2000])

## Opcional: uma pasta inteira

In [ ]:
# PASTA = ROOT / "minhas_imagens"
# for caminho in sorted(PASTA.iterdir()):
#     if caminho.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
#         r = extrair_pose(caminho, salvar_em=ROOT / "diagnostics" / "poses" / f"{caminho.stem}.json")
#         print(f"{caminho.name}: {r['people_detected']} pessoa(s)")

In [ ]:
# !bash rename_images.sh "{ROOT}/Data/jiujitsu_images_out_of_sample/open_guard/dela-riva/nogi/"

In [ ]:
PASTA = ROOT / "Data" / "jiujitsu_images_out_of_sample"
ds = !ls "{PASTA}"

for d in ds:
    !bash rename_images.sh "{PASTA}/{d}/"

In [ ]:
from pathlib import Path

PASTA = ROOT / "Data" / "jiujitsu_images_out_of_sample"
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

imagens = sorted(p for p in PASTA.rglob("*") if p.suffix.lower() in EXTS)
print(f"{len(imagens)} imagens encontradas")

def proxima_imagem():
    for i, imagem in enumerate(imagens, 1):
        rel = imagem.relative_to(PASTA)  # ex.: guarda/image_0003.jpeg
        saida = ROOT / "diagnostics" / "poses" / rel.parent / f"{imagem.stem}.json"
        saida.parent.mkdir(parents=True, exist_ok=True)

        resultado = extrair_pose(imagem, salvar_em=saida)
        print(f"[{i}/{len(imagens)}] {rel}: {resultado['people_detected']} pessoa(s). JSON em {saida}")
        desenhar_pose(imagem, resultado)
        yield

passo = proxima_imagem()

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

PASTA = ROOT / "Data" / "jiujitsu_images_out_of_sample"
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

imagens = sorted(
    p for p in PASTA.rglob("*")
    if p.suffix.lower() in EXTS and ".ipynb_checkpoints" not in p.parts
)

for i, imagem in enumerate(imagens, 1):
    rel = imagem.relative_to(PASTA)
    saida = ROOT / "diagnostics" / "poses" / rel.parent / f"{imagem.stem}.json"
    saida.parent.mkdir(parents=True, exist_ok=True)

    resultado = extrair_pose(imagem, salvar_em=saida)
    print(f"[{i}/{len(imagens)}] {rel}: {resultado['people_detected']} pessoa(s)")

    img = desenhar_pose(imagem, resultado)
    if isinstance(img, np.ndarray):  # caso devolva um array (OpenCV/NumPy)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(str(rel))
        plt.show()
    elif img is not None:            # PIL Image, Figure do matplotlib etc.
        display(img)
    else:                            # a função já desenhou com matplotlib
        plt.show()

In [ ]:
count = 0
images_to_correct = []
for i, imagem in enumerate(imagens, 1):
    rel = imagem.relative_to(PASTA)
    saida = ROOT / "diagnostics" / "poses" / rel.parent / f"{imagem.stem}.json"
    saida.parent.mkdir(parents=True, exist_ok=True)

    resultado = extrair_pose(imagem, salvar_em=saida)
    if resultado['people_detected'] == 1:
        
        print(f"[{i}/{len(imagens)}] {rel}: {resultado['people_detected']} pessoa(s)")
        img = desenhar_pose(imagem, resultado)
        if isinstance(img, np.ndarray):  # caso devolva um array (OpenCV/NumPy)
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.axis("off")
            plt.title(str(rel))
            plt.show()
        elif img is not None:            # PIL Image, Figure do matplotlib etc.
            display(img)
        else:                            # a função já desenhou com matplotlib
            plt.show()

        count +=1
        images_to_correct.append(f"{rel}")

In [ ]:
print(count)